## Use the generate_chat.py and try to pivot the chat messages

In [20]:
import pandas as pd
from generator.generate_chat import *

df = generate_chat_data_v1(num_to_generate=100)
df.head()

,room,username,message,timestamp
0,Room4,UserB,1. Bravo6,2026-03-30 15:39:30.832410
1,Room4,UserB,2. N/A5,2026-03-30 15:39:30.839410
2,Room4,UserB,3. NUMPY2,2026-03-30 15:39:30.845410
3,Room4,UserB,4. None3,2026-03-30 15:39:30.855410
4,Room4,UserB,5. Time to Completion7: 2026-03-30T15:42:26.85...,2026-03-30 15:39:30.856410


### Simple naive merge

In [21]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name", "Timestamp"], message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    to_return = {}
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values

    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [22]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-03-30 16:11:05.050410,Tango10,TEST8,NUMPY5,None7,ETA10: 2026-03-30T16:20:11.085410
1,Room1,2026-03-30 16:28:25.205410,Bravo7,N/A5,TST3,N/A7,Time to Completion7: 2026-03-30T16:31:38.233410
2,Room1,2026-03-30 17:13:39.499410,Delta7,ESCORT3,SCIKIT7,N/A2,Time to Completion9: 2026-03-30T17:13:55.512410
3,Room1,2026-03-30 19:14:32.420410,Tango2,TEST7,TEST10,N/A9,TTC7: 2026-03-30T19:17:38.445410
4,Room1,2026-03-30 19:25:47.467410,Delta6,ESCORT8,PANDAS2,None4,TTC2: 2026-03-30T19:26:46.502410
...,...,...,...,...,...,...,...
95,Room5,2026-03-30 21:21:20.471410,Echo1,SMACK3,TEST5,N/A9,TTC10: 2026-03-30T21:28:24.488410
96,Room5,2026-03-30 22:12:37.893410,Delta5,TEST7,SCIKIT4,None5,ETA7: 2026-03-30T22:22:31.919410
97,Room5,2026-03-30 22:33:31.098410,Delta7,ESCORT10,TEST5,N/A1,ETA6: 2026-03-30T22:39:06.134410
98,Room5,2026-03-30 22:39:06.148410,Bravo2,ESCORT9,TEST9,None2,ETA1: 2026-03-30T22:47:19.173410


### Simple naive merge v2

In [23]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name"], time_column="Timestamp", message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## Add the timestamp column to the order by
    if order_by is None:
        order_by = [time_column]
    elif time_column in order_by:
        pass
    else:
        order_by.append(time_column)

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    ## Create our to return
    to_return = {}

    ## Add the order by column(s) to our results,
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values
    ## Add the prefixes that we are looking for the the results
    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [24]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], time_column="timestamp", message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-03-30 16:11:05.050410,Tango10,TEST8,NUMPY5,None7,ETA10: 2026-03-30T16:20:11.085410
1,Room1,2026-03-30 16:28:25.205410,Bravo7,N/A5,TST3,N/A7,Time to Completion7: 2026-03-30T16:31:38.233410
2,Room1,2026-03-30 17:13:39.499410,Delta7,ESCORT3,SCIKIT7,N/A2,Time to Completion9: 2026-03-30T17:13:55.512410
3,Room1,2026-03-30 19:14:32.420410,Tango2,TEST7,TEST10,N/A9,TTC7: 2026-03-30T19:17:38.445410
4,Room1,2026-03-30 19:25:47.467410,Delta6,ESCORT8,PANDAS2,None4,TTC2: 2026-03-30T19:26:46.502410
...,...,...,...,...,...,...,...
95,Room5,2026-03-30 21:21:20.471410,Echo1,SMACK3,TEST5,N/A9,TTC10: 2026-03-30T21:28:24.488410
96,Room5,2026-03-30 22:12:37.893410,Delta5,TEST7,SCIKIT4,None5,ETA7: 2026-03-30T22:22:31.919410
97,Room5,2026-03-30 22:33:31.098410,Delta7,ESCORT10,TEST5,N/A1,ETA6: 2026-03-30T22:39:06.134410
98,Room5,2026-03-30 22:39:06.148410,Bravo2,ESCORT9,TEST9,None2,ETA1: 2026-03-30T22:47:19.173410


### Try to perform a better merge, that isn't so naive about the timestamp and order

In [25]:

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

new_df = None
for i, prefix in enumerate(prefixes):
    tmpdf = df[df["message"].str.startswith(prefix)].copy()
    tmpdf[new_columns[i]] = tmpdf["message"].str.slice(len(prefix)).values
    tmpdf = tmpdf.drop(columns=["message"])

    if new_df is None:
        new_df = tmpdf
    else:
        new_df = pd.merge(new_df, tmpdf, on=["room", "timestamp", "username"], how="outer")

display(new_df)


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,UserC,2026-03-30 16:11:05.050410,Tango10,NaN,NaN,NaN,NaN
1,Room1,UserC,2026-03-30 16:11:05.064410,NaN,TEST8,NaN,NaN,NaN
2,Room1,UserC,2026-03-30 16:11:05.074410,NaN,NaN,NUMPY5,NaN,NaN
3,Room1,UserC,2026-03-30 16:11:05.077410,NaN,NaN,NaN,None7,NaN
4,Room1,UserC,2026-03-30 16:11:05.085410,NaN,NaN,NaN,NaN,ETA10: 2026-03-30T16:20:11.085410
...,...,...,...,...,...,...,...,...
477,Room5,UserE,2026-03-30 22:47:19.175410,Bravo2,NaN,NaN,NaN,NaN
478,Room5,UserE,2026-03-30 22:47:19.181410,NaN,N/A6,NaN,NaN,NaN
479,Room5,UserE,2026-03-30 22:47:19.191410,NaN,NaN,SCIKIT8,NaN,NaN
480,Room5,UserE,2026-03-30 22:47:19.206410,NaN,NaN,NaN,N/A8,NaN


### Write a function to try and fix the match, to be a little better

In [26]:
def convert_messages_to_ten_line(msg_df,
            prefixes=["1. ", "2. ", "3. ", "4. ", "5. ",
                      "6. ", "7. ", "8. ", "9. ", "10. "],
            new_columns=["msg_1", "msg_2", "msg_3", "msg_4", "msg_5",
                         "msg_6", "msg_7", "msg_8", "msg_9", "msg_10"],
            best_matches=False,
            msg_col="Message", timestmp_col="Timestamp",
            groupby_cnt_col="msg_cnt",
            match_group_col="match_group", time_diff_col="time_diff",
            groupby=["Room Name", "User Name"]):

    ## First go ahead and create the new columns and strip off our prefixes
    new_df = msg_df.copy()
    ## Loop through the prefixes and create the new columns
    for i, prefix in enumerate(prefixes):
        new_df[new_columns[i]] = new_df[new_df[msg_col].str.startswith(prefix)][msg_col].str.slice(len(prefix))

    ## Drop the message column, since we don't need it anymore
    new_df = new_df.drop(columns=[msg_col])

    ## Build our groupby (that includes the timestamp column)
    full_groupby = groupby.copy()
    full_groupby.append(timestmp_col)

    ## Add a column for our groupby count
    new_df[groupby_cnt_col] = 0

    ## GroupBy, first build a dic for our group by
    agg_dict = {}
    for col in new_df.columns:
        if not (col in full_groupby):
          agg_dict[col] = "first"
    agg_dict[groupby_cnt_col] = "count"
    new_df = new_df.groupby(full_groupby).agg(agg_dict).reset_index()


    ## Add the new columns we need for building / calculating matches for our dataframes
    new_df[time_diff_col] = [{} for _ in range(len(new_df))]
    new_df[match_group_col] = 0

    ## If we have rows that equal the number of message we were looking for, then set them aside
    complete_df = new_df[new_df[groupby_cnt_col] == len(prefixes)]

    new_df = new_df[new_df[groupby_cnt_col] < len(prefixes)]
    print("\n")
    print(f"==================== partial df {len(new_df)} records ====================")
    display(new_df)

    ## ======== Calculate the time difference between matching rows ========
    ## Loop through all of the rows and calculate the different time differences between the
    ##     rows that were grouped together
    for index, row in new_df.iterrows():
        #print(f"index: {index}")
        time_differnces = {}
        for index2, row2 in new_df[index + 1:].iterrows():
            #print(f"index2: {index2}")
            if index != index2:
                correct_group = True
                for col in groupby:
                    #print(f"{index}.{row[col]} != {index2}.{row2[col]} ({row[col] != row2[col]})")
                    if row2[col] != row[col]:
                        correct_group = False
                        break

                if correct_group:
                    time_differnces[index2] = (row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000
                    new_df.at[index, match_group_col] += 1
                    new_df.at[index, time_diff_col][index2] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)
                    new_df.at[index2, match_group_col] += 1
                    new_df.at[index2, time_diff_col][index] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)

    ## Take any records that had no matches and move them to the complete dataframe
    tmp_df = new_df[new_df[match_group_col] == 0]
    complete_df = pd.concat([complete_df, tmp_df])

    ## Drop the records where there was no match found
    new_df = new_df[new_df[match_group_col] > 0]

    print("\n")
    print(f"==================== unmatched/complete df {len(complete_df)} records, with timedifferences ====================")
    display(complete_df)

    print("\n")
    print(f"==================== partial df {len(new_df)} records, with timedifferences ====================")
    display(new_df)


    ## ======== Loop through all of our unmatched records and see about ========
    ## Setup our merged dataframe
    merged = {}
    for column in new_df.columns:
        merged[column] = []
    rows_inspected = []
    for index, row in new_df.iterrows():
        ## If we've already used this row, then move along
        if index not in rows_inspected:
            ## Matching Row
            time_differnces = row[time_diff_col]
            min_time_diff = min(time_differnces.values())
            key = [key for key, value in time_differnces.items() if value == min_time_diff][0]
            match_row = new_df.loc[key]

            #rows_inspected.append(index)

    return None

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]
convert_messages_to_ten_line(
    df, prefixes=prefixes, new_columns=new_columns, 
    groupby=["room", "username"], timestmp_col="timestamp", msg_col="message"
)



==================== partial df 482 records ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-03-30 21:10:57.388410,Charlie1,NaN,NaN,NaN,NaN,1,{},0
1,Room1,UserA,2026-03-30 21:10:57.398410,NaN,SMACK3,NaN,NaN,NaN,1,{},0
2,Room1,UserA,2026-03-30 21:10:57.403410,NaN,NaN,TEST6,NaN,NaN,1,{},0
3,Room1,UserA,2026-03-30 21:10:57.418410,NaN,NaN,NaN,None4,NaN,1,{},0
4,Room1,UserA,2026-03-30 21:10:57.430410,NaN,NaN,NaN,NaN,Time to Completion1: 2026-03-30T21:20:12.430410,1,{},0
...,...,...,...,...,...,...,...,...,...,...,...
477,Room5,UserE,2026-03-30 22:47:19.175410,Bravo2,NaN,NaN,NaN,NaN,1,{},0
478,Room5,UserE,2026-03-30 22:47:19.181410,NaN,N/A6,NaN,NaN,NaN,1,{},0
479,Room5,UserE,2026-03-30 22:47:19.191410,NaN,NaN,SCIKIT8,NaN,NaN,1,{},0
480,Room5,UserE,2026-03-30 22:47:19.206410,NaN,NaN,NaN,N/A8,NaN,1,{},0




==================== unmatched/complete df 0 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group




==================== partial df 482 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-03-30 21:10:57.388410,Charlie1,NaN,NaN,NaN,NaN,1,"{1: 10.0, 2: 15.0, 3: 30.0, 4: 42.0, 5: 114713...",24
1,Room1,UserA,2026-03-30 21:10:57.398410,NaN,SMACK3,NaN,NaN,NaN,1,"{0: 10.0, 2: 5.0, 3: 20.0, 4: 32.0, 5: 1147123...",24
2,Room1,UserA,2026-03-30 21:10:57.403410,NaN,NaN,TEST6,NaN,NaN,1,"{0: 15.0, 1: 5.0, 3: 15.0, 4: 27.0, 5: 1147118...",24
3,Room1,UserA,2026-03-30 21:10:57.418410,NaN,NaN,NaN,None4,NaN,1,"{0: 30.0, 1: 20.0, 2: 15.0, 4: 12.0, 5: 114710...",24
4,Room1,UserA,2026-03-30 21:10:57.430410,NaN,NaN,NaN,NaN,Time to Completion1: 2026-03-30T21:20:12.430410,1,"{0: 42.0, 1: 32.0, 2: 27.0, 3: 12.0, 5: 114709...",24
...,...,...,...,...,...,...,...,...,...,...,...
477,Room5,UserE,2026-03-30 22:47:19.175410,Bravo2,NaN,NaN,NaN,NaN,1,"{468: 18985515.0, 469: 18985511.0, 470: 189855...",13
478,Room5,UserE,2026-03-30 22:47:19.181410,NaN,N/A6,NaN,NaN,NaN,1,"{468: 18985521.0, 469: 18985517.0, 470: 189855...",13
479,Room5,UserE,2026-03-30 22:47:19.191410,NaN,NaN,SCIKIT8,NaN,NaN,1,"{468: 18985531.0, 469: 18985527.0, 470: 189855...",13
480,Room5,UserE,2026-03-30 22:47:19.206410,NaN,NaN,NaN,N/A8,NaN,1,"{468: 18985546.0, 469: 18985542.0, 470: 189855...",13


In [27]:
def convert_messages_to_ten_line(msg_df,
            prefixes=["1. ", "2. ", "3. ", "4. ", "5. ",
                      "6. ", "7. ", "8. ", "9. ", "10. "],
            new_columns=["msg_1", "msg_2", "msg_3", "msg_4", "msg_5",
                         "msg_6", "msg_7", "msg_8", "msg_9", "msg_10"],
            best_matches=False,
            msg_col="Message", timestmp_col="Timestamp",
            groupby_cnt_col="msg_cnt",
            match_group_col="match_group", time_diff_col="time_diff",
            groupby=["Room Name", "Username"]):
    """
    Convert our messages, into a ten line format.  Prefixes and newcolumns are defaulted,
        but you can adjust them to whatever format you want.
        Best matches will only return those matches that are smallest
    """

    ## First go ahead and create the new columns and strip off our prefixes
    new_df = msg_df.copy()
    ## Loop through the prefixes and create the new columns
    for i, prefix in enumerate(prefixes):
        new_df[new_columns[i]] = new_df[new_df[msg_col].str.startswith(prefix)][msg_col].str.slice(len(prefix))

    ## Drop the message column, since we don't need it anymore
    new_df = new_df.drop(columns=[msg_col])
    print(f"==================== new df {len(new_df)} records ====================")
    #display(new_df)

    ## Build our groupby (that includes the timestamp column)
    full_groupby = groupby.copy()
    full_groupby.append(timestmp_col)

    ## Add a column for our groupby count
    new_df[groupby_cnt_col] = 0

    ## GroupBy, first build a dic for our group by
    agg_dict = {}
    for col in new_df.columns:
        if not (col in full_groupby):
          agg_dict[col] = "first"
    agg_dict[groupby_cnt_col] = "count"
    new_df = new_df.groupby(full_groupby).agg(agg_dict).reset_index()
    #print("\n")
    #print(f"==================== groupedby df {len(new_df)} records ====================")
    #display(new_df)

    ## Add the new columns we need for building / calculating matches for our dataframes
    new_df[time_diff_col] = [{} for _ in range(len(new_df))]
    new_df[match_group_col] = 0

    ## If we have rows that equal the number of message we were looking for, then set them aside
    complete_df = new_df[new_df[groupby_cnt_col] == len(prefixes)]
    print(f"==================== complete df {len(complete_df)} records ====================")
    #display(complete_df)

    new_df = new_df[new_df[groupby_cnt_col] < len(prefixes)]
    #print("\n")
    print(f"==================== partial df {len(new_df)} records ====================")
    #display(new_df)

    ## ======== Calculate the time difference between matching rows ========
    ## Loop through all of the rows and calculate the different time differences between the
    ##     rows that were grouped together
    for index, row in new_df.iterrows():
        for index2, row2 in new_df[index + 1:].iterrows():
            #print(f"index2: {index2}")
            if index != index2:
                correct_group = True
                for col in groupby:
                    #print(f"{index}.{row[col]} != {index2}.{row2[col]} ({row[col] != row2[col]})")
                    if row2[col] != row[col]:
                        correct_group = False
                        break

                if correct_group:
                    new_df.at[index, match_group_col] += 1
                    new_df.at[index, time_diff_col][index2] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)
                    new_df.at[index2, match_group_col] += 1
                    new_df.at[index2, time_diff_col][index] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)

    ## Take any records that had no matches and move them to the complete dataframe
    tmp_df = new_df[new_df[match_group_col] == 0]
    print(f"==================== partial, with no matches df {len(tmp_df)} records ====================")
    complete_df = pd.concat([complete_df, tmp_df])

    ## Drop the records where there was no match found
    new_df = new_df[new_df[match_group_col] > 0]
    print(f"==================== partial, with matches df {len(new_df)} records ====================")

    #print("\n")
    #print(f"==================== unmatched/complete df {len(complete_df)} records, with timedifferences ====================")
    #display(complete_df)

    #print("\n")
    #print(f"==================== partial df {len(new_df)} records, with timedifferences ====================")
    #display(new_df)


    ## ======== Loop through all of our unmatched records and see about ========
    ## Setup our merged dataframe
    merged = {}
    for column in new_df.columns:
        merged[column] = []
    rows_inspected = []
    for index, row in new_df.iterrows():
        ## If we've already used this row, then move along
        if index not in rows_inspected:
            ## Matching Row
            time_differnces = row[time_diff_col]
            min_time_diff = min(time_differnces.values())
            key = [key for key, value in time_differnces.items() if value == min_time_diff][0]
            match_row = new_df.loc[key]

            #rows_inspected.append(index)

    return None


# convert_messages_to_ten_line(df, prefixes=prefixes, new_columns=new_columns)
convert_messages_to_ten_line(
    df, prefixes=prefixes, new_columns=new_columns, 
    groupby=["room", "username"], timestmp_col="timestamp", msg_col="message"
)

==================== new df 500 records ====================
==================== complete df 0 records ====================
==================== partial df 482 records ====================
==================== partial, with no matches df 0 records ====================
==================== partial, with matches df 482 records ====================


In [29]:
from generator.tenlinesamplegenerator import *

tmp = TenLineSampleGenerator(appendId=True)
fake_ten_line = tmp.generate_sample_data(100)
fake_ten_line.head()

,Room Name,Username,Message,Timestamp
0,s,falcon,1. Taco11_1,2026-03-30 00:00:00.000000Z
1,s,falcon,2. DANCE_1,2026-03-30 00:00:00.000000Z
2,s,falcon,3. PRACTICE_1,2026-03-30 00:00:00.000000Z
3,s,falcon,4. PEACH_1,2026-03-30 00:00:00.000000Z
4,s,falcon,5. N/A_1,2026-03-30 00:00:00.000000Z


In [30]:
from parser.tenlineparser import *

parser = TenLineParser()
results = parser.parse(fake_ten_line)
results[results["msg_cnt"] < 10]

KeyError: 5

In [ ]:
results.iloc[102]['time_diff']